# Logistic regression
Logistic regressions of amplification class on somatic mutation burden, tumor type, batch, sex, age at diagnosis, overall mutation burden, and either LP or P mutation burden.  4 models are evaluated, reflecting 2x dependent variables (amp vs nonamp and ec vs chr) and 2x independent variable parameterizations (lp or p):

## RESULTS

LP mutation burden negatively associated with ecDNA vs chromosomal amp.

| dv | n | iv | p |sign|
|----|----|---|---|----|
|amp vs na|1749|all|0.74|+|
||1749|lp|0.51|-|
|ec vs chr|232|all|0.16|+|
||232|lp|0.007*|-|

No significant differences in P mutation burden using just the CBTN subset.

| dv | n | iv | p |sign|
|----|----|---|---|----|
|amp vs na|970|all|0.30|-|
||970|p|0.92|-|
|ec vs chr|140|all|0.70|-|
||140|p|0.15|-|

Note that n is smaller for these models than for MWU tests because we drop NAs and use only a subset of tumor types.

In [ ]:
#import cyvcf2
import pandas as pd
import numpy as np
import os
from pathlib import Path
#import sklearn
import matplotlib.pyplot as plt

import sys
sys.path.append('../src')
from data_imports import *
from somatic_snv_functions import *

OUT_DIR=Path('out')
OUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
df = merge_ecDNA_counts(import_biosamples(),
                   reindex_counts(read_manifest(),get_variant_count_df())
                  )
df.head()
# write df, including duplicates, then drop duplicates
path=OUT_DIR/'biosample_mutation_burden.tsv'
df.to_csv(path,sep='\t')

df = df[df.in_snv_set]
df.head()

## Test
logistic regression  
forest plot

In [ ]:
import matplotlib.pyplot as plt
#from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [ ]:
def logistic_regression(df,label,predictors):
    # Regress ecDNA status on other independent variables.
    # df: DataFrame from merge_ecDNA_counts above.
    # predictor (iterable): column names of independent variables.
    # label (str): column name of dependent variable.

    # Subset data on cancer_types
    df = df.copy()
    if 'batch' in predictors:
        df['batch'] = df.index.str.startswith('SJ') # SJ or CBTN sets
    df = df[[label,'cancer_type','amplicon_class']+predictors]
    df = df.dropna()
    target_types = get_target_tumor_types(df)
    print(target_types)
    df = df.drop(columns='amplicon_class')
    df = df[df["cancer_type"].isin(target_types)].copy()
    print(df[label].value_counts())
    for col in ['age_at_diagnosis','all_counts','pathogenic_counts','likely_pathogenic_counts']:
        if col in df.columns:
            df[col] = (df[col] - df[col].mean()) / df[col].std(ddof=1)
    for col in ['amplified','ecDNA']:
        if col in df.columns:
            df[col] = df[col].astype(float)
    X = pd.get_dummies(df.drop(columns=label), drop_first=True,dtype=float)
    print(X.columns)
    X = X.astype(float)
    #X = X.to_numpy(dtype=float)
    Xc = X - X.mean(axis=0)
    X = sm.add_constant(X)
    y=df[label].astype(float)
    #y = df[label].to_numpy(dtype=float)

    assert not X.isna().any().any()
    assert not y.isna().any()
    
    #model = sm.Logit(y, X).fit_regularized('l1') # regularization doesn't work in sm.Logit. Known bug apparently.
    #model = sm.GLM(y,X,family=sm.families.Binomial()).fit()
    model = sm.Logit(y,X).fit()
    return model

def forest_plot_regression(model):
    params = model.params
    conf = model.conf_int()
    conf["OR"] = params
    conf.columns = ["2.5%", "97.5%", "OR"]
    conf = np.exp(conf) 
    conf = conf.loc[[c for c in reversed(conf.index) if c != "const"]]
    
    # ---- Forest plot ----
    # TODO add batch, tumor burden to forest plot
    plt.figure(figsize=(6, 4))
    plt.errorbar(conf["OR"], conf.index, 
                 xerr=[conf["OR"] - conf["2.5%"], conf["97.5%"] - conf["OR"]],
                 fmt="o", color="black", ecolor="gray", elinewidth=2, capsize=4)
    plt.axvline(x=1, color="red", linestyle="--")
    plt.xlabel("Odds Ratio (log scale)")
    plt.xscale("log")
    plt.tight_layout()
    
    return

In [ ]:
## Model including all and lp mutations in the same model.
X1 = logistic_regression(df,'amplified',['sex','age_at_diagnosis','batch','all_counts','likely_pathogenic_counts'])
#X1.summary()

In [ ]:
forest_plot_regression(X1)
plt.savefig('out/forest_regression_amp_vs_na.svg')

In [ ]:
X2 = logistic_regression(df[df.amplified],'ecDNA',['sex','age_at_diagnosis','batch','all_counts','likely_pathogenic_counts'])
#X2.summary()

In [ ]:
forest_plot_regression(X2)
plt.savefig('out/forest_regression_ec_vs_ch.svg')

In [ ]:
X3 = logistic_regression(df,'amplified',['sex','age_at_diagnosis','all_counts','pathogenic_counts'])
X3.summary()

In [ ]:
X4 = logistic_regression(df[df.amplified],'ecDNA',['sex','age_at_diagnosis','all_counts','pathogenic_counts'])
X4.summary()

## scratch

In [ ]:
def tune_regression_regularization():
    alphas = np.logspace(-4, 1, 20)  # adjust range if needed
    results = []
    for a in alphas:
        model = sm.GLM(
            y,
            X,
            family=sm.families.Binomial()
        ).fit_regularized(method='elastic_net',alpha=a,L1_wt=1.0, maxiter=10_000) # L1 = 1 -> lasso
        #print(model.summary())
        results.append((a,model.params))
    return results

In [ ]:
def sm_logistic_regression(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())

    return model, X, y

In [ ]:
def sm_logistic_regression_ecDNA_vs_intra(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df = df[df['amplicon_class'] != 'no_amplification'] 
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())

    return model, X, y

In [ ]:
def sklearn_logistic_regression(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    y = df_encoded['amplicon_binary']
    model = LogisticRegression()
    model.fit(X, y)
    #print("Coefficients:", model.coef_)
    #print("Intercept:", model.intercept_)
    #print("Feature names:", X.columns.tolist())
    coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
    }).sort_values("Coefficient", ascending=False)
    print(coef_df)
    return model, X, y

In [ ]:
sklearn_logistic_regression(df_all)

In [ ]:
def sklearn_logistic_regression_ecDNA_vs_intra(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df = df[df['amplicon_class'] != 'no_amplification'] 
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    y = df_encoded['amplicon_binary']
    model = LogisticRegression()
    model.fit(X, y)
    #print("Coefficients:", model.coef_)
    #print("Intercept:", model.intercept_)
    #print("Feature names:", X.columns.tolist())
    coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
    }).sort_values("Coefficient", ascending=False)
    print(coef_df)
    return model, X, y

In [ ]:
m,x,y = sklearn_logistic_regression_ecDNA_vs_intra(df_all)

In [ ]:
m.coef_

In [ ]:
def plot_logistic_by_tumor_type(df):
    """
    Plot logistic regression curves for each tumor type.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing columns:
        - 'amplicon_class' (0 for no amplification, 1 for ecDNA/intrachromosomal)
        - 'all_counts'
        - 'cancer_type'
    """
    df["amplicon_binary"] = df["amplicon_class"].replace({
        "no amplification": 0,
        "ecDNA": 1,
        "intrachromosomal": 1
    })
    
    scaler = StandardScaler()
    df["somatic_variant_scaled"] = scaler.fit_transform(df[["all_counts"]])

    cancer_types = df["cancer_type"].unique()

    plt.figure(figsize=(8, 6))

    for ttype in cancer_types:
        subset = df[df["cancer_type"] == ttype]
        X = subset[["somatic_variant_scaled"]]
        y = subset["amplicon_binary"]

       
        if len(subset[y == 1]) > 0 and len(subset[y == 0]) > 0:
            model = LogisticRegression()
            model.fit(X, y)

            x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
            y_pred = model.predict_proba(x_range)[:, 1]

            plt.plot(x_range, y_pred, label=ttype)

    plt.xlabel("Somatic variant counts (scaled)")
    plt.ylabel("Probability of amplification")
    plt.title("Logistic regression by cancer type")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_logistic_by_tumor_type(df)

In [ ]:
def logistic_regression_all_types(df):
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)

    cancer_types = df['cancer_type'].unique()

    plt.figure(figsize=(10, 7))

    for ctype in cancer_types:
        sub = df[df['cancer_type'] == ctype]
        # クラスが1種類しかない場合はスキップ
        if len(sub['amplicon_binary'].unique()) < 2:
            continue

        X = sub[['all_counts']]
        y = sub['amplicon_binary']

        model = LogisticRegression()
        model.fit(X, y)

        # 曲線を描画するためのX範囲を作成
        x_vals = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
        y_pred = model.predict_proba(x_vals)[:, 1]

        # データ点（薄く表示）
        plt.scatter(X, y, alpha=0.25, s=15)

        # 各cancer typeごとのロジスティック曲線
        plt.plot(x_vals, y_pred, linewidth=2, label=ctype)

    plt.xlabel('Somatic Variant Counts')
    plt.ylabel('Binary (0=no amplification, 1=ecDNA/intrachromosomal)')
    plt.title('Logistic Regression Curves by Cancer Type')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
logistic_regression_all_types(df)

In [ ]:
import matplotlib.pyplot as plt

def logistic_regression_all_types_ecDNA_vs_intra(df):
    # ecDNAとintrachromosomalのみ対象
    df = df[df['amplicon_class'].isin(['ecDNA', 'intrachromosomal'])].copy()
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)

    cancer_types = df['cancer_type'].unique()

    plt.figure(figsize=(10, 7))

    for ctype in cancer_types:
        sub = df[df['cancer_type'] == ctype]
        # クラスが1種類しかない場合はスキップ
        if len(sub['amplicon_binary'].unique()) < 2:
            continue

        X = sub[['all_counts']]
        y = sub['amplicon_binary']

        model = LogisticRegression()
        model.fit(X, y)

        # 曲線を描画するためのX範囲を作成
        x_vals = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
        y_pred = model.predict_proba(x_vals)[:, 1]

        # データ点（薄く表示）
        plt.scatter(X, y, alpha=0.25, s=15)

        # 各cancer typeごとのロジスティック曲線
        plt.plot(x_vals, y_pred, linewidth=2, label=ctype)

    plt.xlabel('Somatic Variant Counts')
    plt.ylabel('Binary (0=intrachr, 1=ecDNA)')
    plt.title('Logistic Regression Curves by Cancer Type')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
logistic_regression_all_types_ecDNA_vs_intra(df)